In [ ]:
!pip install wandb timm yacs gdown

In [2]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

In [3]:
!git clone https://$github_token@github.com/Visoura/Visoura-ReID.git

Cloning into 'Visoura-ReID'...
remote: Enumerating objects: 1630, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 1630 (delta 1), reused 4 (delta 1), pack-reused 1623 (from 1)
Receiving objects: 100% (1630/1630), 113.62 MiB | 44.16 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [4]:
wandb_token = user_secrets.get_secret("WANDB_AUTH_TOKEN")

import wandb
wandb.login(key=wandb_token)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: divomagdy (divomagdy-ain-shams-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os, gdown

os.makedirs("/kaggle/working/weights", exist_ok=True)
WEIGHTS = "/kaggle/working/weights/vit_small_pretrained.pth"
if not os.path.exists(WEIGHTS):
    gdown.download(id="1V3WnN7TbXRfLDCepblfBk5tNYAOG4hL5", output=WEIGHTS, quiet=False)
print(WEIGHTS, os.path.getsize(WEIGHTS) / 1e6, "MB")

In [ ]:
#   baseline | baseline_2x | baseline_2x_l2norm | koleo_2x | uniformity_2x | variance_2x
EXPS = ["baseline", "baseline_2x"] 
SEED = 42
NUM_WORKERS = 2                       

import subprocess, time, os
os.chdir("/kaggle/working/Visoura-ReID/transreid_pytorch")
os.makedirs("/kaggle/working/logs", exist_ok=True)

procs = {}
for gpu, exp in enumerate(EXPS):
    name = f"{exp}_s{SEED}"
    cmd = ["python", "train.py", "--config_file", f"configs/wahdan/{exp}.yml",
           "SOLVER.SEED", str(SEED),
           "MODEL.DEVICE_ID", f"('{gpu}')",        # train.py maps this to CUDA_VISIBLE_DEVICES
           "DATALOADER.NUM_WORKERS", str(NUM_WORKERS),
           "WANDB.RUN_NAME", name,
           "WANDB.PROJECT", "Cross-ReID-PersonViT",
           "OUTPUT_DIR", f"log/transreid/market/{name}"]
    logf = open(f"/kaggle/working/logs/{name}.log", "w")
    procs[name] = (subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT), logf)
    print(f"started {name} on GPU {gpu}")

def last_line(name):
    with open(f"/kaggle/working/logs/{name}.log") as f:
        lines = f.read().strip().splitlines()
    return lines[-1][:160] if lines else ""

while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(120)
    for name, (p, _) in procs.items():
        state = "running" if p.poll() is None else f"exit {p.returncode}"
        print(f"[{name}] {state} | {last_line(name)}")

for name, (p, logf) in procs.items():
    logf.close()
    print(f"{name}: exit code {p.returncode}  (log: /kaggle/working/logs/{name}.log)")